feature_repo/features.py — Define HR attrition features once, use everywhere

In [ ]:
import sys
!{sys.executable} -m pip install feast

In [ ]:
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64
from datetime import timedelta

employee = Entity(name="employee_number", description="An employee in the HR dataset")

employee_features_source = FileSource(
    path="s3://your-ml-data-lake/curated/training_datasets/employee_features.parquet",
    timestamp_field="event_timestamp",
)

employee_attrition_features = FeatureView(
    name="employee_attrition",
    entities=[employee],
    ttl=timedelta(days=365),
    schema=[
        # Engineered satisfaction aggregate
        Field(name="overall_satisfaction", dtype=Float32),
        # Binned income and age
        Field(name="annual_income_band", dtype=Int64),
        Field(name="age_group", dtype=Int64),
        # Career risk signals
        Field(name="promotion_stagnation", dtype=Float32),
        Field(name="career_velocity", dtype=Float32),
        # Binary indicators
        Field(name="long_commute", dtype=Int64),
        Field(name="stable_manager", dtype=Int64),
        Field(name="overtime", dtype=Int64),
        # Raw numeric features
        Field(name="job_level", dtype=Int64),
        Field(name="performance_rating", dtype=Int64),
        Field(name="years_at_company", dtype=Int64),
        Field(name="total_working_years", dtype=Int64),
    ],
    source=employee_features_source,
)

Training: fetch historical features at label time (prevents leakage)

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path="feature_repo/")

# labels_df must have columns: employee_number, event_timestamp, attrition
training_df = store.get_historical_features(
    entity_df=labels_df,
    features=[
        "employee_attrition:overall_satisfaction",
        "employee_attrition:annual_income_band",
        "employee_attrition:age_group",
        "employee_attrition:promotion_stagnation",
        "employee_attrition:career_velocity",
        "employee_attrition:long_commute",
        "employee_attrition:stable_manager",
        "employee_attrition:overtime",
        "employee_attrition:job_level",
        "employee_attrition:performance_rating",
        "employee_attrition:years_at_company",
        "employee_attrition:total_working_years",
    ],
).to_df()

Serving: fetch latest features for a live employee_number

In [ ]:
store = FeatureStore(repo_path="feature_repo/")

features = store.get_online_features(
    features=[
        "employee_attrition:overall_satisfaction",
        "employee_attrition:promotion_stagnation",
        "employee_attrition:career_velocity",
        "employee_attrition:overtime",
        "employee_attrition:job_level",
        "employee_attrition:years_at_company",
    ],
    entity_rows=[{"employee_number": 1042}],
).to_dict()